# Systematic Feature Selection: Annual & Decadal Trends in Cattle Mortality

## Objective

Identify the **optimal predictors of annual cattle mortality trends** and **decadal climate shifts** in USDA Regions 4 & 6 using long-term aggregations (1984-2025).

## Scientific Rationale

**Annual/Decadal Analysis** captures:
- Long-term climate change signals
- Secular trends in heat stress
- Decade-specific vulnerabilities
- Growing season patterns

**Key Differences from Weekly Analysis**:
- Smaller sample size (~40-80 annual observations vs. ~2,000 weekly)
- Less noise, clearer trends
- Different feature space (annual means, extremes, trends)
- Higher expected R² (0.60-0.80 vs. 0.45-0.55)

## Hypotheses

**H1**: Linear year trend will explain substantial variance (climate change signal)

**H2**: Decadal models will show shifting heat stress patterns (2010s-2020s more severe)

**H3**: Annual VPD and extreme heat days will be strongest predictors

**H4**: Simpler models will perform better (fewer features needed for annual aggregations)

---

In [6]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Statistical modeling (optional - for advanced diagnostics)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("⚠ statsmodels not available - some advanced features will be skipped")

# Configuration
import sys
sys.path.append('../../')
from config import (
    PROCESSED_WEEKLY_DIR, MASK_FILE, CATTLE_DATA_FILE,
    CATTLE_REGION_IDS, SEASONS
)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
np.random.seed(42)

# Output directory
OUTPUT_DIR = Path('../../figures/feature_selection_annual')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded successfully")

Libraries loaded successfully


## 1. Load Weekly Data and Aggregate to Annual

In [7]:
# Load weekly data
print("Loading weekly climate data...")
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

week_dates = ds_night['week'].values

# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function
def compute_regional_mean(data, state_ids, state_mask):
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']

# Compute weekly regional metrics
climate_weekly = []
for region_num, state_ids in [(4, region_4_ids), (6, region_6_ids)]:
    df = pd.DataFrame({
        'week_ending': pd.to_datetime(week_dates),
        'region': region_num,
        'daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], state_ids, state_mask).values,
        'daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], state_ids, state_mask).values,
        'nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], state_ids, state_mask).values,
        'nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], state_ids, state_mask).values,
        'vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], state_ids, state_mask).values,
        'vpd_max': compute_regional_mean(ds_vpd['vpd_max'], state_ids, state_mask).values,
    })
    df['year'] = df['week_ending'].dt.year
    climate_weekly.append(df)

climate_weekly_df = pd.concat(climate_weekly, ignore_index=True)

print(f"Weekly data: {len(climate_weekly_df)} region-weeks")

# Aggregate to annual
print("\nAggregating to annual metrics...")

annual_climate = climate_weekly_df.groupby(['region', 'year']).agg({
    'daytime_hours_above_30': ['mean', 'max', 'sum', lambda x: np.percentile(x, 95)],
    'daytime_hours_above_35': ['mean', 'max', 'sum', lambda x: np.percentile(x, 95)],
    'nighttime_hours_above_21': ['mean', 'max', 'sum'],
    'nighttime_hours_above_24': ['mean', 'max', 'sum'],
    'vpd_mean': ['mean', 'max', lambda x: np.percentile(x, 95)],
    'vpd_max': ['mean', 'max', lambda x: np.percentile(x, 95)],
}).reset_index()

# Flatten column names
annual_climate.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                          for col in annual_climate.columns.values]

# Rename lambda columns
annual_climate = annual_climate.rename(columns={
    'daytime_hours_above_30_<lambda>': 'daytime_hours_above_30_p95',
    'daytime_hours_above_35_<lambda>': 'daytime_hours_above_35_p95',
    'vpd_mean_<lambda>': 'vpd_mean_p95',
    'vpd_max_<lambda>': 'vpd_max_p95',
})

print(f"Annual climate data: {len(annual_climate)} region-years")
print(f"Features: {len([c for c in annual_climate.columns if c not in ['region', 'year']])}")

Loading weekly climate data...
Weekly data: 4382 region-weeks

Aggregating to annual metrics...
Annual climate data: 84 region-years
Features: 20


In [8]:
# Load and aggregate cattle data to annual
print("\nLoading and aggregating cattle data...")

cattle_data = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
cattle_data['year'] = cattle_data['date'].dt.year

# Annual totals by region
annual_cattle = cattle_data.groupby('year').agg({
    'region_4_beef_dairy': 'sum',
    'region_6_beef_dairy': 'sum'
}).reset_index()

# Reshape to long format
annual_r4 = annual_cattle[['year']].copy()
annual_r4['region'] = 4
annual_r4['total_beef_dairy'] = annual_cattle['region_4_beef_dairy']

annual_r6 = annual_cattle[['year']].copy()
annual_r6['region'] = 6
annual_r6['total_beef_dairy'] = annual_cattle['region_6_beef_dairy']

annual_cattle_long = pd.concat([annual_r4, annual_r6], ignore_index=True)

# Merge climate and cattle
annual_data = pd.merge(annual_cattle_long, annual_climate, on=['region', 'year'], how='inner')
annual_data = annual_data.dropna()

print(f"\nMerged annual data: {len(annual_data)} region-years")
print(f"Year range: {annual_data['year'].min()}-{annual_data['year'].max()}")
print(f"\nSample:")
print(annual_data.head())


Loading and aggregating cattle data...

Merged annual data: 84 region-years
Year range: 1984-2025

Sample:
   year  region  total_beef_dairy  daytime_hours_above_30_mean  \
0  1984       4            1039.3                 1.854323e+13   
1  1985       4             911.3                 2.367272e+13   
2  1986       4            1021.4                 3.405557e+13   
3  1987       4             907.5                 2.645522e+13   
4  1988       4             751.7                 2.929630e+13   

   daytime_hours_above_30_max  daytime_hours_above_30_sum  \
0                8.527284e+13                9.642479e+14   
1                1.088396e+14                1.230981e+15   
2                1.471169e+14                1.770889e+15   
3                1.386173e+14                1.375672e+15   
4                1.334875e+14                1.552704e+15   

   daytime_hours_above_30_<lambda_0>  daytime_hours_above_35_mean  \
0                       7.261764e+13                 1.0829

## 2. Create Annual/Decadal Features

In [9]:
# Add trend features
annual_data['year_centered'] = annual_data['year'] - annual_data['year'].mean()
annual_data['year_squared'] = annual_data['year_centered'] ** 2

# Decade indicators
annual_data['decade'] = (annual_data['year'] // 10) * 10
annual_data['decade_1980s'] = (annual_data['decade'] == 1980).astype(int)
annual_data['decade_1990s'] = (annual_data['decade'] == 1990).astype(int)
annual_data['decade_2000s'] = (annual_data['decade'] == 2000).astype(int)
annual_data['decade_2010s'] = (annual_data['decade'] == 2010).astype(int)
annual_data['decade_2020s'] = (annual_data['decade'] == 2020).astype(int)

# Define feature sets
target_var = 'total_beef_dairy'

# Climate features (from aggregation)
climate_features = [col for col in annual_data.columns 
                    if col not in ['region', 'year', 'total_beef_dairy', 'year_centered', 
                                  'year_squared', 'decade'] and not col.startswith('decade_')]

# Trend features
trend_features = ['year_centered', 'year_squared']

# Decade features
decade_features = [col for col in annual_data.columns if col.startswith('decade_')]

# Regional
regional_features = ['region']

# All features
all_annual_features = climate_features + trend_features + decade_features + regional_features

print(f"\nFeature breakdown:")
print(f"  Climate aggregations: {len(climate_features)}")
print(f"  Trend features: {len(trend_features)}")
print(f"  Decade indicators: {len(decade_features)}")
print(f"  Regional: {len(regional_features)}")
print(f"  TOTAL: {len(all_annual_features)}")


Feature breakdown:
  Climate aggregations: 20
  Trend features: 2
  Decade indicators: 5
  Regional: 1
  TOTAL: 28


## 3. Train/Test Split and Baseline Models

Due to small sample size, we'll use:
- **Train**: 1984-2010 (~27 years per region = 54 observations)
- **Test**: 2011-2025 (~15 years per region = 30 observations)
- **Cross-validation**: 5-fold CV on training set

In [10]:
# Prepare data
model_data = annual_data[all_annual_features + [target_var, 'year']].dropna().copy()

# Train/test split
train_mask = model_data['year'] <= 2010
test_mask = model_data['year'] > 2010

X_train = model_data.loc[train_mask, all_annual_features]
y_train = model_data.loc[train_mask, target_var]
X_test = model_data.loc[test_mask, all_annual_features]
y_test = model_data.loc[test_mask, target_var]

print(f"Train set: {len(X_train)} observations")
print(f"Test set: {len(X_test)} observations")

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures standardized")

Train set: 54 observations
Test set: 30 observations

Features standardized
